# 02 - LangGraph Basics

A chain is a straight line. Real applications need **loops** (try a tool, look at the
result, maybe try another) and **branches** (route this question to a different path).

LangGraph gives you a state machine: **state** that flows through **nodes** connected by
**edges**. Modern LangChain agents are built on it — `create_agent` returns a LangGraph
graph — so understanding this notebook is what makes notebook 03 readable rather than magic.

By the end you will be able to:
1. Define graph state with a `TypedDict` and a reducer.
2. Add nodes and edges, and route conditionally.
3. Compile, invoke, stream and visualise a graph.
4. Add persistent memory with a checkpointer and `thread_id`.

> **No API key needed.** Most of this notebook is plain Python functions — that is the
> point. A node does not have to call an LLM.

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

import langgraph
print("langgraph ready")

## 1. State

Every graph has one shared **state** object. Each node receives it and returns a partial
update, which LangGraph merges in.

By default a returned key **overwrites** the old value. That is wrong for a message list —
you want to *append*. A **reducer** controls the merge, and `add_messages` is the built-in
one for conversations.

In [ ]:
class CounterState(TypedDict):
    value: int              # no reducer -> each node overwrites this
    log: Annotated[list, lambda old, new: old + new]     # reducer -> appends

def double(state: CounterState) -> dict:
    return {"value": state["value"] * 2, "log": [f"doubled to {state['value'] * 2}"]}

def add_ten(state: CounterState) -> dict:
    return {"value": state["value"] + 10, "log": [f"added 10 -> {state['value'] + 10}"]}

builder = StateGraph(CounterState)
builder.add_node("double", double)
builder.add_node("add_ten", add_ten)
builder.add_edge(START, "double")
builder.add_edge("double", "add_ten")
builder.add_edge("add_ten", END)

graph = builder.compile()
result = graph.invoke({"value": 5, "log": []})

print("final value:", result["value"])
print("log:")
for line in result["log"]:
    print("  ", line)

Notice the difference: `value` was replaced at each step (5 → 10 → 20), while `log`
accumulated. **That distinction is the most common source of LangGraph bugs** — a message
list without `add_messages` silently loses history.

## 2. The anatomy of a graph

| Piece | What it is |
|---|---|
| **State** | a `TypedDict` describing everything that flows through |
| **Node** | a function `state -> partial state update` |
| **Edge** | "after this node, go to that one" |
| **Conditional edge** | a function that *decides* which node comes next |
| `START` / `END` | the entry and exit points |
| `.compile()` | validates the wiring and returns a runnable graph |

A node is an ordinary Python function. It does not have to touch an LLM — it can query a
database, call an API, or do arithmetic.

## 3. Conditional edges — routing

This is the thing a chain cannot do. A routing function inspects the state and returns the
**name** of the next node.

In [ ]:
class TicketState(TypedDict):
    text: str
    category: str
    response: str

def classify(state: TicketState) -> dict:
    """In a real app this would be an LLM call with structured output."""
    text = state["text"].lower()
    if any(w in text for w in ["charge", "refund", "invoice", "bill"]):
        category = "billing"
    elif any(w in text for w in ["error", "crash", "bug", "broken"]):
        category = "technical"
    else:
        category = "general"
    return {"category": category}

def handle_billing(state: TicketState) -> dict:
    return {"response": "Routed to the billing team. Expect a reply within 4 hours."}

def handle_technical(state: TicketState) -> dict:
    return {"response": "Routed to engineering. Please attach your log file."}

def handle_general(state: TicketState) -> dict:
    return {"response": "Thanks for getting in touch - a human will reply shortly."}

def route(state: TicketState) -> Literal["billing", "technical", "general"]:
    """The routing function returns the NAME of the next node."""
    return state["category"]

builder = StateGraph(TicketState)
builder.add_node("classify", classify)
builder.add_node("billing", handle_billing)
builder.add_node("technical", handle_technical)
builder.add_node("general", handle_general)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route)      # <- the branch
for node in ["billing", "technical", "general"]:
    builder.add_edge(node, END)

support_graph = builder.compile()

for text in ["I was charged twice this month!",
             "The export button crashes the app",
             "Do you have an office in Beirut?"]:
    out = support_graph.invoke({"text": text, "category": "", "response": ""})
    print(f"{out['category']:10} | {out['response']}")

## 4. Visualising the graph

A compiled graph can draw itself. Reading the picture is the fastest way to spot wiring
mistakes.

In [ ]:
print(support_graph.get_graph().draw_mermaid())

Paste that into [mermaid.live](https://mermaid.live) to see it rendered. In Jupyter with
`pygraphviz` installed you can also use:

```python
from IPython.display import Image
Image(support_graph.get_graph().draw_mermaid_png())
```

## 5. Loops — what makes it an *agent*

A conditional edge can point **backwards**. That creates a cycle, which is how an agent
keeps calling tools until it is done. This is the structure `create_agent` builds for you.

In [ ]:
class RetryState(TypedDict):
    attempts: int
    solved: bool
    history: Annotated[list, lambda old, new: old + new]

def attempt(state: RetryState) -> dict:
    n = state["attempts"] + 1
    solved = n >= 3                       # pretend it succeeds on the third try
    return {"attempts": n, "solved": solved,
            "history": [f"attempt {n}: {'success' if solved else 'failed'}"]}

def should_retry(state: RetryState) -> Literal["attempt", "__end__"]:
    if state["solved"]:
        return END
    if state["attempts"] >= 5:            # ALWAYS bound your loops
        return END
    return "attempt"

builder = StateGraph(RetryState)
builder.add_node("attempt", attempt)
builder.add_edge(START, "attempt")
builder.add_conditional_edges("attempt", should_retry)

retry_graph = builder.compile()
out = retry_graph.invoke({"attempts": 0, "solved": False, "history": []})

for line in out["history"]:
    print(" ", line)
print("\nsolved:", out["solved"], "after", out["attempts"], "attempts")

> **Always bound your loops.** An agent whose exit condition never fires will burn tokens
> until it hits a recursion limit. LangGraph defaults to 25 steps, but write your own
> ceiling — as the `attempts >= 5` line does above.

## 6. Streaming

`.invoke()` waits for the whole run. `.stream()` yields as it goes, which is what you need
for a UI that shows progress.

In [ ]:
print("stream_mode='updates' - what each node changed:\n")
for chunk in support_graph.stream(
        {"text": "The export button crashes", "category": "", "response": ""},
        stream_mode="updates"):
    for node_name, update in chunk.items():
        print(f"  {node_name:10} -> {update}")

print("\nstream_mode='values' - the whole state after each step:\n")
for snapshot in support_graph.stream(
        {"text": "The export button crashes", "category": "", "response": ""},
        stream_mode="values"):
    print("  ", snapshot)

| `stream_mode` | Yields | Use for |
|---|---|---|
| `"updates"` | just what each node changed | logging, showing progress |
| `"values"` | the full state after each step | debugging |
| `"messages"` | LLM tokens as they generate | a chat UI that types |

## 7. Memory — checkpointers and `thread_id`

This is how "the agent remembers" actually works, and it replaces LangChain 0.x's
`ConversationBufferMemory`.

A **checkpointer** saves the state after every step. A **`thread_id`** names a conversation.
Same `thread_id` means the state is loaded back; a different one starts fresh.

In [ ]:
from langchain.messages import HumanMessage, AIMessage

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]     # the reducer that appends

def respond(state: ChatState) -> dict:
    turn = len([m for m in state["messages"] if isinstance(m, HumanMessage)])
    return {"messages": [AIMessage(f"(reply {turn}) I have {len(state['messages'])} messages in context.")]}

builder = StateGraph(ChatState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)

chat = builder.compile(checkpointer=InMemorySaver())      # <- memory switched on

alice = {"configurable": {"thread_id": "alice"}}

for text in ["Hello", "What is my name?", "Still there?"]:
    out = chat.invoke({"messages": [HumanMessage(text)]}, config=alice)
    print(f"user: {text:20} agent: {out['messages'][-1].content}")

print("\nTotal messages remembered on alice's thread:", len(out["messages"]))

In [ ]:
# A different thread_id is a different conversation - nothing is shared.
bob = {"configurable": {"thread_id": "bob"}}
out_bob = chat.invoke({"messages": [HumanMessage("Hi")]}, config=bob)

print("bob's thread :", len(out_bob["messages"]), "messages")
print("alice's thread:", len(chat.get_state(alice).values["messages"]), "messages")

> **`InMemorySaver` disappears when the process exits.** For anything real use a durable
> checkpointer — `langgraph-checkpoint-sqlite` for a single machine,
> `langgraph-checkpoint-postgres` for a server. The code change is one line.

### Inspecting and rewinding

Because every step is checkpointed, you can look at the state, and even go back to an
earlier point and take a different path.

In [ ]:
state = chat.get_state(alice)
print("next node to run:", state.next)
print("messages so far :", len(state.values["messages"]))

print("\ncheckpoint history (newest first):")
for i, snapshot in enumerate(chat.get_state_history(alice)):
    print(f"  {i}: {len(snapshot.values['messages'])} messages")
    if i >= 4:
        break

---
## Exercises

1. Remove the reducer from `log` in section 1 (make it a plain `list`). Run it again and
   explain what happens to the log.
2. Add a fourth category `"sales"` to the support graph, with its own node and keyword
   rule. Redraw the mermaid diagram to confirm the wiring.
3. In the retry graph, change `attempt` so it never succeeds and remove the
   `attempts >= 5` guard. What happens, and what error do you get? Now put the guard back.
4. Build a two-node graph where node A fetches a number and node B decides — via a
   conditional edge — whether to go back to A or finish. Bound it at 10 iterations.
5. Add a `user_name` field to `ChatState` and have the node fill it from the first message.
   Confirm it persists across turns on the same `thread_id` but not across different ones.
6. **Stretch:** install `langgraph-checkpoint-sqlite` and swap `InMemorySaver` for
   `SqliteSaver`. Restart the kernel, re-run with the same `thread_id`, and confirm the
   conversation survived.

## Next

Notebook 03 builds a real tool-using agent — first with `create_agent` in four lines, then
hand-built as a `StateGraph` so you can see that those four lines produce exactly the graph
you just learned to draw.